# Plotting

We present some examples for how to use the basic plotting functions in the python interface for GVEC. Note that
these are not intended for advanced plotting, but for quickly visualising some basic properties.

All plotting functions are found in `gvec.plotting`, and most of them can be called directly as methods of a `state` object.

We will use the same elliptic stellarator as in the tutorial [](./020_stellarator) for the plotting examples.

In [ ]:
import gvec

In [ ]:
params = {
    "ProjectName": "test_plotting",
    "which_hmap": 1,
    "PhiEdge": 1.0,
    "iota": {"type": "polynomial", "coefs": [0.625, 0.35]},
    "pres": {
        "type": "polynomial",
        "coefs": [1.0, -1.0],
        "scale": 1000.0,
    },
    "nfp": 3,
    "X1_b_cos": {(0, 0): 3.0, (1, 0): 1.0, (1, 1): 0.4},
    "X2_b_sin": {(1, 0): 1.0, (1, 1): -0.4, (0, 1): -0.25},
    "init_average_axis": True,
    "sgrid_nElems": 2,
    "X1_mn_max": [3, 3],
    "X2_mn_max": [3, 3],
    "LA_mn_max": [3, 3],
    "X1X2_deg": 5,
    "LA_deg": 5,
    "totalIter": 1000,
    "minimize_tol": 1.0e-6,
}


try:
    state = gvec.find_state()
except ValueError:
    run = gvec.run(params)
    state = run.state

Note that all 1D and 2D plotting functions return the `matplotlib` figure and axis objects, or a figure and array of axis objects if multiple subplots are output.

# 1D plots

## Radial profiles

The first 1D plot is for radial profiles, i.e. scalar functions of $\rho$. Note that only scalar quantities can be evaluated for these plots. The defaults will give you rotational transform $\iota$, pressure $p$, and the toroidal current $I_{\mathrm{tor}}$ and poloidal current $I_{\mathrm{pol}}$.

In [ ]:
f, ax = state.plot_radial_profile()

one can add the rationals up to a certain order to the plot of the rotational transform

In [ ]:
gvec.plotting.add_iota_rationals(state, ax)
f

## Magnetic axis

Properties along the magnetic axis can be plotted with `plot_on_axis`. Since some derived quantities cannot be evaluated on axis,
this will automatically use a quadratic extrapolation from `rho=[1.1e-4, 2.2e-4, 3.3e-4]` to compute the values at `rho=0`.

Note that we have specified the `subplot_grid` here, which defines the `row,col` of the layout of the subplots. This is not required unless you want a specific grid layout. As with the previous set of plots, the $x$-axis will automatically be shared between the columns.

In [ ]:
f, ax = state.plot_on_axis(
    quantities=["mod_B", "mod_J"],
    subplot_grid=[2, 1],
)

# 2D plotting

## Poloidal slice plots

For plotting poloidal slices we can lock or unlock the $X^1$ and $X^2$ values on the $x$ and $y$ axes by specifying `share_axis=True/False` (by default this is `True`). By default we plot $|\mathbf{B}|$, and add contours of fixed $\rho$ and $\vartheta^\star$ values (PEST coordinates) in white.

In [ ]:
f, ax = state.plot_poloidal_plane(zeta=4)

We can also plot contours of other quantities, e.g. the pressure, and change or leave out the contours of the coordinates:

In [ ]:
f, ax = state.plot_poloidal_plane(
    quantity="p",
    share_axis=True,
    rho_contours=2,
    rho_contours_color="red",
    theta_contours=0,
    zeta=4,
)

## Flux surface plots

For plotting values on specific flux surfaces we use `plot_on_flux_surface`. Note that multiple flux surfaces can be shown at once by specifying a list or `numpy.ndarray` of flux surfaces labels. By default only the last closed flux surface (GVEC boundary) is plotted.

We also plot in Boozer coordinates, but this can be changed to either PEST or $(\vartheta,\zeta)$ coordinates by setting `sfl="pest"` or `sfl=None` respectively.

Finally, note that `kwargs` specific for `matplotlib.pyplot.subplots` can be handed to any 1D or 2D plotting function by the dictionary input, here we change the size of the plot using this feature.

In [ ]:
f, ax = state.plot_on_flux_surface(
    rho=[0.3, 0.6], plot_kwargs={"figsize": (6, 4)}
)

Note that we can also evaluate different values on the same flux surface by specifying the `quantities` rather than multiple `rho` values (currently we cannot do both). The quantities should be scalar values, as with the 1D plots.

In the plot below we also display filled contours by changing the `style`, and show the quantities in regular $(\vartheta,\zeta)$ coordinates rather than straight-field-line coordinates.

In [ ]:
f, ax = state.plot_on_flux_surface(
    quantities=["mod_B", "Jac"],
    subplot_grid=[2, 1],
    sfl=None,
    style="filled-contour",
)

# Help

Before moving on to the 3D plotting functions, note that for the list of inputs for any function you can always call help on the plotting function

In [ ]:
help(state.plot_radial_profile)

# 3D plotting

For 3D plotting we use [plotly](https://plotly.com/python/) as a backend.
To plot the boundary we only need the state file and the resolution of the plot. Note that we can also specify the quantity we want to plot with
the `quantities` keyword, by default $\|B\|$ will be plotted on the boundary.

Unlike the previous plots which return a `figure` and `axis`, 3D plots only return a `plotly.Figure` object.

Note that you may need a version of `plotly<6.0` in order to display the plots in a jupyter notebook.

In [ ]:
fig = state.plot_3d_surface()
fig.show()

The value and position of the surface being plotting can be changed in the same way as the previous plots.

If for some reason _plotly_ does not `show`, there is an optional input, `to_file` (default `None`),
which can be set to a string to write the plot to a file in your current working directory.

In [ ]:
fig = state.plot_3d_surface("L_gradB", rho=0.5, ntheta=31, nzeta=41)
fig.show()